# SLM-JA-G2P v11 推論デモ

学習済みLoRAアダプタ（Hugging Face）を使って日本語テキストの読み（カタカナ）を推定する。
GPU（T4で可）推奨。CPUでも動作する（ビーム4では1文あたり十数秒かかることがある。`read(text, num_beams=1)` で高速化可）。

構成は論文の最終システムと同一: **文分割＋カナ制約デコード＋ビーム探索（幅4）**。


In [ ]:
%%capture
!pip uninstall -y torchao  # Colab同梱のtorchao 0.10はtransformers新版と衝突するため先に除去
!pip install -q "transformers>=4.46" peft accelerate


In [ ]:
# ===== モデル読込（HF Hubからアダプタ取得→マージ） =====
import re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from peft import PeftModel

REPO = "t2-fujii/slm-ja-g2p-v11-lora"
BASE = "sbintuitions/sarashina2.2-1b-instruct-v0.1"
dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if dev == "cuda" else torch.float32
print("device:", dev, "| dtype:", dtype)
tok = AutoTokenizer.from_pretrained(REPO)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype)
model = PeftModel.from_pretrained(model, REPO).merge_and_unload().to(dev).eval()

SYSTEM = ("あなたは日本語テキストの読み推定器です。与えられた文の読みを、カタカナのみで1行で出力してください。"
          "発音通りに書くこと: 助詞の「は」は「ワ」、「へ」は「エ」、「を」は「オ」。数字や記号も読みをカタカナで書くこと。")
OK = re.compile(r"^[ァ-ヶー、]+$")
allowed = sorted({i for i in range(len(tok)) if OK.match(tok.decode([i]))} | {tok.eos_token_id})
allowed_t = torch.tensor(allowed, device=dev)
class KanaMask(LogitsProcessor):
    def __call__(self, ids, scores):
        mask = torch.full_like(scores, float("-inf"))
        mask[:, allowed_t] = 0.0
        return scores + mask
SPLIT = re.compile(r"(?<=[。！？])(?![。！？])")

def read(text, num_beams=4):
    """日本語テキスト→カタカナ読み（論文の最終システム構成: 文分割+カナ制約+ビーム4）"""
    outs = []
    for part in (p for p in SPLIT.split(text) if p.strip()):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": part}]
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True).to(dev)
        with torch.no_grad():
            out = model.generate(**enc, do_sample=False, num_beams=num_beams,
                                 length_penalty=0.0, early_stopping=True,
                                 max_new_tokens=min(256, 4 * len(part)),
                                 pad_token_id=tok.eos_token_id,
                                 logits_processor=LogitsProcessorList([KanaMask()]))
        outs.append(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return "".join(outs)
print("ready")


In [ ]:
# ===== デモ（辞書G2Pが誤りやすい文） =====
samples = [
    "著名人が1993年の一日を語った。",
    "ブータン国内に鉄道は通っていない。",
    "研究所の所長は世論の動向を重視した。",
    "次の駅で電車を降ります。確かに窓の外は雨だった。",
    "電話番号は090の1234の5678です。",
]
for s in samples:
    print(s)
    print(" →", read(s))
    print()


In [ ]:
# ===== 自由入力 =====
text = "ここに読みを知りたい日本語テキストを入れてください。"
print(read(text))
